In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder, StandardScaler, MinMaxScaler
import warnings
warnings.filterwarnings('ignore')

# ------ 1. LOAD DATA ------
df = pd.read_csv('/kaggle/input/datasets/zenishadevani/dataset/ai4i2020 (1).csv')
print("Shape:", df.shape)
print(df.head(3))

# ------ 2. DROP USELESS COLUMNS ------
df.drop(columns=['UDI', 'Product ID'], inplace=True)

# ------ 3. ENCODE CATEGORICAL ------
le = LabelEncoder()
df['Type_enc'] = le.fit_transform(df['Type'])   # L=0, M=1, H=2
# Keep original for potential interaction features
type_dummies = pd.get_dummies(df['Type'], prefix='Type')
df = pd.concat([df, type_dummies], axis=1)
df.drop(columns=['Type'], inplace=True)

# ------ 4. DOMAIN-SPECIFIC FEATURES ------
# Temperature difference (process - air) is a known failure indicator
df['Temp_diff'] = df['Process temperature [K]'] - df['Air temperature [K]']

# Power = Torque × Angular velocity (ω = RPM × 2π/60)
df['Power_W'] = df['Torque [Nm]'] * (df['Rotational speed [rpm]'] * 2 * np.pi / 60)

# Tool wear rate normalized by rotational speed
df['Wear_per_RPM'] = df['Tool wear [min]'] / (df['Rotational speed [rpm]'] + 1e-6)

# Mechanical stress proxy: Torque / RPM
df['Stress_proxy'] = df['Torque [Nm]'] / (df['Rotational speed [rpm]'] + 1e-6)

# Heat dissipation failure condition (from paper: Temp_diff < 8.6K and RPM < 1380)
df['HDF_condition'] = ((df['Temp_diff'] < 8.6) & (df['Rotational speed [rpm]'] < 1380)).astype(int)

# Power failure condition (from paper: Power outside 3500-9000W range)
df['PWF_condition'] = ((df['Power_W'] < 3500) | (df['Power_W'] > 9000)).astype(int)

# Overstrain failure condition (from paper: Tool wear × Torque > threshold per type)
df['OSF_condition_L'] = ((df['Type_L'] == 1) & (df['Tool wear [min]'] * df['Torque [Nm]'] > 11000)).astype(int)
df['OSF_condition_M'] = ((df['Type_M'] == 1) & (df['Tool wear [min]'] * df['Torque [Nm]'] > 12000)).astype(int)
df['OSF_condition_H'] = ((df['Type_H'] == 1) & (df['Tool wear [min]'] * df['Torque [Nm]'] > 13000)).astype(int)
df['OSF_condition'] = df[['OSF_condition_L','OSF_condition_M','OSF_condition_H']].max(axis=1)

# Wear × Torque (key interaction)
df['Wear_x_Torque'] = df['Tool wear [min]'] * df['Torque [Nm]']

# ------ 5. DEFINE TARGET ------
# Binary classification: Machine failure (0 or 1)
TARGET = 'Machine failure'

# Drop individual failure mode columns (they're sub-labels, not features)
FAILURE_COLS = ['TWF', 'HDF', 'PWF', 'OSF', 'RNF']
feature_cols = [c for c in df.columns if c not in [TARGET] + FAILURE_COLS]

X = df[feature_cols]
y = df[TARGET]

print(f"\nFeatures: {list(X.columns)}")
print(f"Class distribution:\n{y.value_counts()}")
print(f"\nX shape: {X.shape}, y shape: {y.shape}")

# ------ 6. SCALING ------
scaler = StandardScaler()
X_scaled = pd.DataFrame(scaler.fit_transform(X), columns=X.columns)

print("\n✅ Feature Engineering V1 complete.")
print(f"Total features: {X_scaled.shape[1]}")

Shape: (10000, 14)
   UDI Product ID Type  Air temperature [K]  Process temperature [K]  \
0    1     M14860    M                298.1                    308.6   
1    2     L47181    L                298.2                    308.7   
2    3     L47182    L                298.1                    308.5   

   Rotational speed [rpm]  Torque [Nm]  Tool wear [min]  Machine failure  TWF  \
0                    1551         42.8                0                0    0   
1                    1408         46.3                3                0    0   
2                    1498         49.4                5                0    0   

   HDF  PWF  OSF  RNF  
0    0    0    0    0  
1    0    0    0    0  
2    0    0    0    0  

Features: ['Air temperature [K]', 'Process temperature [K]', 'Rotational speed [rpm]', 'Torque [Nm]', 'Tool wear [min]', 'Type_enc', 'Type_H', 'Type_L', 'Type_M', 'Temp_diff', 'Power_W', 'Wear_per_RPM', 'Stress_proxy', 'HDF_condition', 'PWF_condition', 'OSF_condition_L'